### Import 

In [1]:
import numpy as np
import json
import random
from tqdm import tqdm
import matplotlib.pyplot as plt
import os

In [2]:
import IPython
import soundfile as sf

### Object365v1

In [191]:
with open('../../data/objects365v1/annotations/objects365_train_text.json', 'r') as f:
    obj_anns = json.load(f)

In [3]:
len(obj_anns['images']), len(obj_anns['annotations'])

(608606, 9621875)

In [7]:
obj_anns['images'][:1]

[{'file_name': 'obj365_train_000000367456.jpg',
  'height': 512,
  'width': 768,
  'id': 367456}]

In [8]:
obj_anns['annotations'][:1]

[{'iscrowd': 0,
  'image_id': 367456,
  'bbox': [562.1600342016, 192.9611816448, 93.19201658880002, 231.1978149376],
  'id': 0,
  'area': 21545.790604959133,
  'category_id': 1}]

In [192]:
num_img = 131072
sub_annots = obj_anns.copy()

file_set = set([])
for i, img in enumerate(sub_annots['images']):
    if 'train' in img['file_name']:
        file_set.add(img['id'])
    # file_set.add(img['file_name']) # + '-' + str(img['id']))
    if len(file_set) > num_img:
        break

sub_imgs = []
for i, img in enumerate(sub_annots['images']):
    if img['id'] in file_set:
        sub_imgs.append(img)
        
sub_anns = []
for i, an in enumerate(sub_annots['annotations']):
    if an['image_id'] in file_set:
        sub_anns.append(an)
        
sub_annots['images'] = sub_imgs
sub_annots['annotations'] = sub_anns

In [10]:
sub_annots['images'][:2]

[{'file_name': 'obj365_train_000000367456.jpg',
  'height': 512,
  'width': 768,
  'id': 367456},
 {'file_name': 'obj365_train_000000693364.jpg',
  'height': 512,
  'width': 768,
  'id': 693364}]

In [18]:
len(sub_annots['annotations'])

1032054

In [193]:
with open('../../data/objects365v1/annotations/objects365_train_131k.json', 'w') as f:
    json.dump(sub_annots, f)

In [35]:
for i in range(4):
    wav,sr = sf.read(f'../obj365v1_audio_v2/349-0-eagle-{i}.wav')
    IPython.display.display(IPython.display.Audio(wav, rate=sr))

In [170]:
with open('../../data/texts/obj365v1_class_texts.json', 'r') as f:
    obj365v1_class_texts = json.load(f)

In [171]:
rep_str = {'suv': 'S U V',
           'cd': 'C D',
           'tv': 'T V',
           'pear': 'peer',
           'deer': 'dear',
           'soccer': 'sucker',
           'hair drier': 'hair dryer',
           'dumbbell': 'dumb bell',
           'hamimelon': 'hammymelon',
           'hotair balloon': 'hot-air balloon',
           'cymbal': 'symbol', }

In [172]:
tts_jsons = []
repeat = 4

for i,ts in enumerate(obj365v1_class_texts):
    for j, t in enumerate(ts):
        if t in rep_str:
            t = rep_str[t]
            
        text = t.replace('  ', ' ')
        text = text.replace(' ', '_')
        text = text.replace('-', '_')
        
        uid = f'{i}-{j}-{text}'
        if repeat == 1:
            tts_jsons.append([uid, t])
        else:
            for m in range(repeat):
                tts_jsons.append([uid + f'-{m}', t])

In [180]:
tts_jsons[:8]

[['0-0-person-0', 'person'],
 ['0-0-person-1', 'person'],
 ['0-0-person-2', 'person'],
 ['0-0-person-3', 'person'],
 ['1-0-sneakers-0', 'sneakers'],
 ['1-0-sneakers-1', 'sneakers'],
 ['1-0-sneakers-2', 'sneakers'],
 ['1-0-sneakers-3', 'sneakers']]

In [173]:
with open('../sample_images/object365v1/tts.json', 'w') as f:
    json.dump(tts_jsons, f)

In [72]:
[t[1] for t in tts_jsons][:12]

['person',
 'person',
 'person',
 'person',
 'sneakers',
 'sneakers',
 'sneakers',
 'sneakers',
 'chair',
 'chair',
 'chair',
 'chair']

In [181]:
miss_utts = 0
miss_textses = []
textses = [[] for i in range(365)]

# with torch.no_grad():
for uid, texts in tqdm(tts_jsons[::-1], ncols=50):
    iid = uid
    f_path = f"../obj365v1_audio_v2/{iid}.wav"

    if not os.path.exists(f_path):
        miss_textses.append([uid, texts])
        miss_utts += 1
        
    wav, sr = sf.read(f_path)
    e = len(wav)
    textses[int(iid.split('-')[0])].append([f"demo/obj365v1_audio_v2/{iid}.wav", 0, e])
        # continue
        
print(miss_utts, miss_textses)

100%|███████| 1516/1516 [00:00<00:00, 7974.00it/s]

0 []


In [183]:
# textses
with open('../../data/audios/obj365v1_class_audio2.json', 'w') as f:
    json.dump(textses, f)

In [104]:
'wine glass'.strip().replace(' ', '')

'wineglass'

In [175]:
[t[1] for t in textses]

['comb', 'cue', 'cue']

In [176]:
textses

[['340-0-comb-3', 'comb'], ['256-0-cue-3', 'cue'], ['256-0-cue-1', 'cue']]

### COCO / LVIS

In [37]:
with open('../../data/texts/lvis_v1_class_texts.json', 'r') as f:
    class_texts = json.load(f)

In [3]:
with open('../../data/texts/coco_class_texts.json', 'r') as f:
    class_texts = json.load(f)

In [4]:
len(class_texts)

80

In [14]:
# rep_str = {'yoghourt': 'yoghurt',  
#            'soupspoon': 'soup spoon',
#            'tv_camera': 'TV camera',
#            'pegleg': 'peg leg',
#            'sportswear': 'sports wear',
#            'deer': 'dear',
#            }

rep_str = {'hair drier': 'hairdryer',
           'tv': 'T V'} 
# rep_str = {}

In [17]:
# rep_str = {}

tts_jsons = []
repeat = 4

for i,ts in enumerate(class_texts):
    for j, t in enumerate(ts):
        if t in rep_str:
            t = rep_str[t]
            
        text = t.replace('  ', ' ')
        text = text.replace(' ', '_')
        text = text.replace('-', '_')
        
        uid = f'{i}-{j}-{text}'
        if repeat == 1:
            tts_jsons.append([uid, t])
        else:
            for m in range(repeat):
                tts_jsons.append([uid + f'-{m}', t])

In [ ]:
yogurt

In [53]:
with open('../sample_images/lvis/tts.json', 'w') as f:
    json.dump(tts_jsons, f)

In [16]:
with open('../sample_images/coco/tts.json', 'w') as f:
    json.dump(tts_jsons, f)

In [30]:
miss_utts = 0
miss_textses = []
num_classes = 80
num_classes = 1203

textses = [[] for i in range(num_classes)]
# audio_dir = 'lvis_audio_v2'
audio_dir = 'coco_audio_v2'

# with torch.no_grad():
for uid, texts in tqdm(tts_jsons[::-1], ncols=50):
    iid = uid
    f_path = f"../{audio_dir}/{iid}.wav"

    # if not os.path.exists(f_path):
    try:
        wav, sr = sf.read(f_path)
        e = len(wav)
        textses[int(iid.split('-')[0])].append([f"demo/{audio_dir}/{iid}.wav", 0, e])
        # continue
    except Exception as e:
        miss_textses.append([uid, texts])
        miss_utts += 1
        
        
print(miss_utts, miss_textses[:2])

100%|█████████| 320/320 [00:00<00:00, 8232.70it/s]

70 [['79-0-toothbrush-3', 'toothbrush'], ['79-0-toothbrush-1', 'toothbrush']]


In [60]:
t2num = {}
for uid,text in miss_textses:
    if text not in t2num:
        t2num[text] = 1
    else:
        t2num[text] += 1
        
    if t2num[text] == 4:
        print(text)

urn
vat
tv set
plaid
armoured combat vehicle
calamary
shears
daikon
racquet
propellor
clayware
pole
plyers
pita
pinecone
pea
parroquet
paroquet
paraquet
parroket
parrakeet
pouffe
kerosine lamp
oar
feedbag
nightclothes
leging clothing
leging
lasagne
laniard
koala bear
koala
doorknocker
jewellery
tee shirt
humous
hoummos
hommos
nargileh
narghile
grater
cincture
earplug
earpiece
dumbbell
doughnut
doorknob
cervid
cymbal
cot
crape
pelmet
valance board
valance
cooky
cullender
cocoanut
hatrack
coatrack
clothes hamper
clipboard
cyder
chile
chilly vegetable
chilly
chilli vegetable
chequebook
daybed
chaise
chaise longue
cantaloup
cannister
charabanc
buoy
breechclout
breechcloth
bandeau
reel
vizor
beret
boeuf food
boeuf
bedpan
barrette
bandanna
baguet
beigel
bagel
haversack
packsack
ax
nebuliser
armour
golosh


In [25]:
for uid, texts in tqdm(tts_jsons[::-1][120:160], ncols=50):
    iid = uid
    f_path = f"../{audio_dir}/{iid}.wav"

    # if not os.path.exists(f_path):
    try:
        wav, sr = sf.read(f_path)
        print(texts)
        IPython.display.display(IPython.display.Audio(wav, rate=sr))
    except Exception as e:
        continue

  0%|                      | 0/40 [00:00<?, ?it/s]

orange


orange


orange


sandwich


...

wine glass


wine glass


wine glass


wine glass


100%|████████████| 40/40 [00:00<00:00, 269.37it/s]
